# Evaluación final única · Prevención de Fraudes (2026)

Este notebook utiliza el **modelo congelado antes de consultar test** por `Prevencion_Fraudes_metodologia_correcta_2026_v3.ipynb`. Recupera el mismo split aleatorio estratificado, verifica hashes SHA256, aplica el umbral elegido con los datos de calibración y calcula resultados del test una sola vez.

**No recalcula ni elige hiperparámetros, modelos ni umbrales.** La base es un **dataset público simulado**, por lo que los resultados no demuestran eficacia en transacciones bancarias reales. Este experimento es una revisión de 2026, no la entrega académica de 2022.

Requiere `outputs/prevencion_fraudes_revision_2026/modelo_congelado_pretest.joblib` y `manifiesto_congelado_pretest.json`; solo cargar `.joblib` de origen confiable. Permite definir `FRAUD_CSV_PATH` y `FRAUD_ARTIFACT_DIR`. No ejecutar una segunda vez con la intención de seleccionar un mejor modelo a partir de los resultados.

In [ ]:
from pathlib import Path
import os, json, hashlib, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
    f1_score, fbeta_score, precision_score, recall_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay)

def sha256_archivo(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(2**20), b''):
            h.update(bloque)
    return h.hexdigest()

ARTEFACTOS = Path(os.environ.get('FRAUD_ARTIFACT_DIR','outputs/prevencion_fraudes_revision_2026'))
RUTA_CSV = Path(os.environ.get('FRAUD_CSV_PATH','data/card_transdata.csv'))
MANIFIESTO = json.loads((ARTEFACTOS/'manifiesto_congelado_pretest.json').read_text(encoding='utf-8'))
assert MANIFIESTO['estado']=='MODELO_CONGELADO_PRETEST'
assert sha256_archivo(RUTA_CSV)==MANIFIESTO['csv_sha256'], 'CSV diferente al utilizado en desarrollo.'
RUTA_MODELO=ARTEFACTOS/'modelo_congelado_pretest.joblib'
assert sha256_archivo(RUTA_MODELO)==MANIFIESTO['modelo_joblib_sha256'], 'Modelo cambiado después del desarrollo.'
print('Manifiesto original:', MANIFIESTO['modelo'], 'CV',MANIFIESTO['cv_folds'], 'iteraciones',MANIFIESTO['search_iter'])
print('CSV y modelo congelado: hashes validados.')


## Recuperación exacta del conjunto reservado y evaluación

Se reconstruye la partición por semilla; el hash de sus índices debe coincidir con el guardado antes de evaluar. Se ejecuta el modelo previamente fijado, con el umbral ya elegido.

In [ ]:
TARGET='fraud'
CONTINUAS=['distance_from_home','distance_from_last_transaction','ratio_to_median_purchase_price']
BINARIAS=['repeat_retailer','used_chip','used_pin_number','online_order']
FEATURES=CONTINUAS+BINARIAS
base=pd.read_csv(RUTA_CSV,usecols=FEATURES+[TARGET])
X=base[FEATURES]
y=base[TARGET].astype(int)
X_train_pool, X_test, y_train_pool, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=MANIFIESTO['semilla'])
_, X_cal, _, y_cal = train_test_split(X_train_pool, y_train_pool, test_size=0.20, stratify=y_train_pool, random_state=MANIFIESTO['semilla']+1)
# Los índices de test fueron comprometidos antes de consultar sus etiquetas.
hash_test = hashlib.sha256(np.asarray(X_test.index,dtype=np.int64).tobytes()).hexdigest()
assert hash_test == MANIFIESTO['test_indices_sha256'], 'Los índices de test no coinciden.'
assert len(X_test)==MANIFIESTO['n_prueba_reservada'] and len(X_cal)==MANIFIESTO['n_calibracion']
RUTA_METRICAS=ARTEFACTOS/'metricas_prueba_final.json'
if RUTA_METRICAS.exists():
    raise FileExistsError('Ya hay resultados de test. Evitar una segunda evaluación selectiva.')
modelo=joblib.load(RUTA_MODELO)  # Archivo joblib propio; nunca cargar fuentes no confiables.
umbral=MANIFIESTO['umbral_f2_calibrado']
prob=modelo.predict_proba(X_test)[:,1]
pred=(prob>=umbral).astype(int)
tn,fp,fn,tp=map(int,confusion_matrix(y_test,pred,labels=[0,1]).ravel())
metricas={
    'tipo':'REVISION_2026_TEST_FINAL_UNICO',
    'dataset_sha256':MANIFIESTO['csv_sha256'],
    'modelo_sha256':MANIFIESTO['modelo_joblib_sha256'],
    'nombre_modelo':MANIFIESTO['modelo'],
    'umbral_elegido_sin_test':umbral,
    'n_entrenamiento':MANIFIESTO['n_entrenamiento'],
    'n_calibracion':MANIFIESTO['n_calibracion'],
    'n_prueba':len(X_test),
    'prevalencia_fraude_prueba':float(y_test.mean()),
    'accuracy':float(accuracy_score(y_test,pred)),
    'precision':float(precision_score(y_test,pred,zero_division=0)),
    'recall':float(recall_score(y_test,pred,zero_division=0)),
    'f1':float(f1_score(y_test,pred,zero_division=0)),
    'f2':float(fbeta_score(y_test,pred,beta=2,zero_division=0)),
    'average_precision':float(average_precision_score(y_test,prob)),
    'roc_auc':float(roc_auc_score(y_test,prob)),
    'TN':tn,'FP':fp,'FN':fn,'TP':tp,
}
ARTEFACTOS.mkdir(parents=True, exist_ok=True)
# Guardar inmediatamente: no se reoptimiza ni se repite el test.
RUTA_METRICAS.write_text(json.dumps(metricas,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([metricas]).T.rename(columns={0:'Valor'}))
print('Reporte de clasificación:')
print(classification_report(y_test,pred,zero_division=0))
print('Métricas finales guardadas en:',RUTA_METRICAS)


In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=[0,1],ax=ax)
ax.set_title('Matriz de confusión — TEST FINAL');plt.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(6,4))
PrecisionRecallDisplay.from_predictions(y_test,prob,ax=ax)
ax.set_title('Precision–Recall — TEST FINAL');plt.tight_layout();plt.show()
fig,ax=plt.subplots(figsize=(6,4))
RocCurveDisplay.from_predictions(y_test,prob,ax=ax)
ax.set_title('ROC — TEST FINAL');plt.tight_layout();plt.show()


## Interpretación y limitaciones

La evaluación se realizó una sola vez sobre una partición independiente de 200.000 registros, manteniendo intactos el modelo y el umbral comprometidos. El dataset **simulado** presenta relaciones potencialmente mucho más sencillas que el fraude financiero real. No existen identificadores de clientes/tarjetas ni marcas temporales para una validación temporal o por grupo. No se evaluaron costos reales, calibración de probabilidades ni deriva de datos. Los resultados se circunscriben al dataset y al protocolo de 2026.